# Session 12 — Individual Bias Audit: Equalized Odds Variant
**Research Methods & Scientific Integrity in AI** — Doctoral Program in Deep Technologies, UNMSM  
**Instructor:** Dr. Loveleen Gaur  
**Student:** Cendik Leonardo Mauricio Orcón  

### What makes this submission different from the exemplar
The professor's exemplar uses **Reweighing** and prioritizes **statistical parity / disparate impact**. This individual variant keeps the same COMPAS dataset and race audit, but intentionally changes the scientific choice:

- **Mitigation:** Fairlearn `ExponentiatedGradient` (**in-processing**)
- **Constraint:** `EqualizedOdds`
- **Primary fairness objective:** reduce both the **TPR gap** and **FPR gap** between racial groups
- **Evaluation:** same baseline, same test design, BEFORE/AFTER comparison, and 10 repeated splits

**No result is preloaded.** All tables and the final report are generated from the values produced by this execution.


## 1. Setup

In [ ]:
# --- One-time setup (Google Colab) ---
!pip -q install aif360 fairlearn "pandas<3"

import warnings, os, urllib.request, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from aif360.datasets import CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric

# This submission uses Fairlearn's in-processing reduction.
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("pandas", pd.__version__, "| toolkits ready.")


In [ ]:
# --- Fetch the COMPAS data that AIF360 expects (not bundled, for licensing reasons) ---
import aif360
compas_dir = os.path.join(os.path.dirname(aif360.__file__), "data", "raw", "compas")
os.makedirs(compas_dir, exist_ok=True)
csv_path = os.path.join(compas_dir, "compas-scores-two-years.csv")
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"

if not os.path.exists(csv_path) or os.path.getsize(csv_path) < 1_000_000:
    urllib.request.urlretrieve(url, csv_path)

size = os.path.getsize(csv_path)
assert size > 1_000_000, f"COMPAS download looks truncated ({size} bytes). Re-run this cell."
print(f"COMPAS data present ({size:,} bytes).")

## 2. Config and audit helpers

One config cell drives the whole notebook, so you can re-run everything for a different protected attribute by editing a single line.

In [ ]:
# ============ CONFIG: edit this, then Runtime > Run all ============
STUDENT_NAME = "Cendik Leonardo Mauricio Orcón"

PROTECTED = "race"
TEST_FRACTION = 0.30

MITIGATION_NAME = "ExponentiatedGradient"
MITIGATION_FAMILY = "in-processing"
FAIRNESS_OBJECTIVE = "Equalized Odds"

# EqualizedOdds constrains both the TPR and FPR gaps.
# 0.02 means we ask the reduction to search for a solution close to a
# 2-percentage-point difference bound during optimization.
EO_BOUND = 0.02

# ExponentiatedGradient's own optimization tolerance.
EG_EPS = 0.01

RUN_ADULT_ALTERNATIVE = False
# ==================================================================

# AIF360 encodes COMPAS as:
# race 1 = Caucasian, 0 = African-American
# sex  1 = Female,    0 = Male
GROUP_LABELS = {
    "race": {1: "Caucasian (privileged)", 0: "African-American (unprivileged)"},
    "sex":  {1: "Female (privileged)",    0: "Male (unprivileged)"},
}

privileged_groups   = [{PROTECTED: 1}]
unprivileged_groups = [{PROTECTED: 0}]

print("Student :", STUDENT_NAME)
print("Auditing:", PROTECTED, "->", GROUP_LABELS[PROTECTED])
print("Mitigation:", MITIGATION_NAME, f"({MITIGATION_FAMILY})")
print("Fairness objective:", FAIRNESS_OBJECTIVE)
print("Equalized-Odds difference bound:", EO_BOUND)


In [ ]:
# --- Helpers so BEFORE and AFTER are always measured the exact same way ---

def make_pred_dataset(base_dataset, model, X, favorable_label):
    """Wrap a standard sklearn model output in an AIF360 dataset."""
    fav_col = list(model.classes_).index(favorable_label)
    out = base_dataset.copy(deepcopy=True)
    out.labels = model.predict(X).reshape(-1, 1)
    out.scores = model.predict_proba(X)[:, fav_col].reshape(-1, 1)
    return out

def make_fairlearn_pred_dataset(base_dataset, favorable_predictions, favorable_label, unfavorable_label):
    """Convert Fairlearn's 1=favorable predictions back to COMPAS label coding.

    Fairlearn's reduction works with binary labels 0/1 and treats 1 as the
    positive class. COMPAS/AIF360 uses 0 as the favorable outcome. We therefore
    train the mitigation on a recoded target (1 = favorable) and then map its
    predictions back to the original AIF360 labels before auditing.
    """
    pred_fav = np.asarray(favorable_predictions).astype(int).ravel()
    original_labels = np.where(pred_fav == 1, favorable_label, unfavorable_label)

    out = base_dataset.copy(deepcopy=True)
    out.labels = original_labels.reshape(-1, 1)
    # ExponentiatedGradient has no predict_proba; these are hard decision scores.
    out.scores = pred_fav.astype(float).reshape(-1, 1)
    return out

def audit(dataset_true, dataset_pred, unpriv, priv):
    """Return performance plus group-fairness metrics.

    For this individual submission, Equalized Odds is the primary objective.
    It requires BOTH the TPR gap and FPR gap to be small.
    """
    cm = ClassificationMetric(
        dataset_true, dataset_pred,
        unprivileged_groups=unpriv,
        privileged_groups=priv
    )

    di = cm.disparate_impact()
    if not np.isfinite(di):
        print(
            "WARNING: disparate impact is not finite. "
            "One group received zero favorable predictions."
        )

    return {
        "Accuracy":                 cm.accuracy(),
        "Disparate impact":         di,
        "Statistical parity diff":  cm.statistical_parity_difference(),
        "Equal opportunity diff":   cm.equal_opportunity_difference(),       # TPR difference
        "False positive rate diff": cm.false_positive_rate_difference(),      # FPR difference
        "Equalized odds gap":       cm.equalized_odds_difference(),           # max(|TPR gap|, |FPR gap|)
        "Average odds diff":        cm.average_odds_difference(),
    }

def fourfifths_flag(di):
    return "PASS (>= 0.8)" if di >= 0.8 else "FAIL (< 0.8): potential disparate impact"

def show(name, m):
    print(name)
    for k, v in m.items():
        print(f"  {k:27s}: {v: .3f}")
    print("  Four-fifths rule           :", fourfifths_flag(m["Disparate impact"]))


## 3. Load the data and pin down the direction of "favorable"

Run the next cell first. The code prints and asserts the favorable-label direction. The interpretation below is based on that verified encoding, not on an assumption.


In [ ]:
dataset = CompasDataset()

train, test = dataset.split([1 - TEST_FRACTION], shuffle=True, seed=RANDOM_STATE)

FAV = dataset.favorable_label      # do NOT hard-code this to 1
UNFAV = dataset.unfavorable_label

print("Label column      :", dataset.label_names)
print("Favorable label   :", FAV)
print("Unfavorable label :", UNFAV)
print("Protected attrs   :", dataset.protected_attribute_names)
print("Train / test sizes:", train.features.shape[0], "/", test.features.shape[0])
print()

# The label column is two_year_recid: 1 = rearrested within two years, 0 = not.
# AIF360 therefore sets favorable_label = 0.  Being predicted NOT to re-offend is the
# outcome that leads to the better real-world consequence (release, lower bail).
assert FAV == 0.0, ("Expected the favorable label to be 0 (did NOT re-offend). "
                    "If this fires, the dataset encoding changed and every metric "
                    "below would silently reverse meaning.")
print("DIRECTION CONFIRMED")
print("  favorable  = label 0 = did NOT re-offend within two years = good for the defendant")
print("  unfavorable= label 1 = rearrested within two years")

### My interpretation of the outcome direction

- **Protected attribute:** `race`
- **Privileged group:** Caucasian (`race = 1`)
- **Unprivileged group:** African-American (`race = 0`)
- **Favorable outcome:** the defendant is predicted **not to re-offend within two years** (`label 0`).
- **Why I recode the target for Fairlearn:** `EqualizedOdds` works with binary labels where `1` is the positive class. For the mitigation only, I recode **favorable = 1** and **unfavorable = 0**. After prediction, I map the result back to COMPAS/AIF360 coding before calculating the audit metrics.


## 4. Measure bias in the raw data

This looks only at the ground-truth labels. No model exists yet, so anything you find here is measured historical inequality, not a modelling artifact. It is also the ceiling problem for everything that follows: you are about to ask a classifier to be fairer than the data it learns from.

In [ ]:
raw = BinaryLabelDatasetMetric(train,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups)
print(f"Disparate impact (data):        {raw.disparate_impact():.3f}")
print(f"Statistical parity diff (data): {raw.mean_difference():.3f}")
print(f"Base rate, unprivileged:        {raw.base_rate(privileged=False):.3f}")
print(f"Base rate, privileged:          {raw.base_rate(privileged=True):.3f}")

## 5. Look at what you are actually feeding the model

Two facts about AIF360's default COMPAS feature set are worth a minute of class discussion before you train anything.

In [ ]:
feat = dataset.feature_names
print("Number of features:", len(feat))
print()
print("Is each protected attribute itself present among the input features?")
for p in dataset.protected_attribute_names:
    print(f"  {p:5s} in features: {p in feat}")
print()

charge_desc = [f for f in feat if f.startswith("c_charge_desc=")]
print(f"c_charge_desc contributes {len(charge_desc)} one-hot columns.")
print()

print("Interpretation:")
print("  - Protected attributes and proxy variables can both carry group information.")
print("  - Deleting race alone is not an Equalized-Odds mitigation.")
print("  - This audit measures group error rates directly rather than assuming")
print("    that 'fairness through unawareness' is sufficient.")


## 6. Train the baseline model

The required baseline is **Logistic Regression** on standardized features. The following cell fits the model and computes all BEFORE metrics directly from the test split.


In [ ]:
scaler  = StandardScaler().fit(train.features)
X_train = scaler.transform(train.features)
X_test  = scaler.transform(test.features)
y_train = train.labels.ravel()

# Completed: create and fit the baseline classifier
baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

# Completed: wrap predictions into AIF360 datasets
train_pred = make_pred_dataset(train, baseline, X_train, FAV)
test_pred  = make_pred_dataset(test,  baseline, X_test,  FAV)

before = audit(test, test_pred, unprivileged_groups, privileged_groups)
show("BEFORE mitigation", before)

## 7. Individual mitigation choice — Exponentiated Gradient + Equalized Odds

The professor's exemplar targets **statistical parity** with Reweighing. I intentionally investigate a different fairness question: **are the model's error rates distributed similarly across racial groups?**

**Selected method:** `ExponentiatedGradient` (**in-processing**)  
**Constraint:** `EqualizedOdds`  
**Primary quantities:** TPR gap and FPR gap  

This choice is defensible in a recidivism-risk setting because both kinds of error can create serious harm. A false alarm can incorrectly assign higher risk to someone who would not re-offend, while a miss represents a different error burden. Equalized Odds therefore asks the model to reduce **both** error-rate disparities rather than equalizing selection rates alone.


In [ ]:
# --- Professor's exemplar method: Reweighing ---
# Not executed in this individual submission.
#
# The exemplar uses Reweighing to target statistical parity / disparate impact.
# My individual scientific choice is different:
#   ExponentiatedGradient + EqualizedOdds
#
# This cell is intentionally left as a documented reference point.
print("Reweighing is NOT executed in this individual variant.")
print("Selected method:", MITIGATION_NAME, "+", FAIRNESS_OBJECTIVE)


In [ ]:
# --- SELECTED METHOD: ExponentiatedGradient + EqualizedOdds ---

# Sensitive feature used by the Fairlearn constraint.
sens_train = train.protected_attributes[
    :, train.protected_attribute_names.index(PROTECTED)
].ravel()

# IMPORTANT:
# COMPAS/AIF360: favorable = 0.
# Fairlearn reductions: binary positive class = 1.
# Recode only for the mitigation so 1 means FAVORABLE.
y_train_fav = (y_train == FAV).astype(int)

constraint = EqualizedOdds(difference_bound=EO_BOUND)

eg = ExponentiatedGradient(
    estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    constraints=constraint,
    eps=EG_EPS,
)

eg.fit(
    X_train,
    y_train_fav,
    sensitive_features=sens_train
)

# ExponentiatedGradient.predict() is randomized by design.
# Fix random_state so this evaluation is reproducible.
pred_fav = eg.predict(X_test, random_state=RANDOM_STATE)

test_pred_mit = make_fairlearn_pred_dataset(
    test,
    pred_fav,
    favorable_label=FAV,
    unfavorable_label=UNFAV
)

after = audit(
    test,
    test_pred_mit,
    unprivileged_groups,
    privileged_groups
)

SELECTED_MITIGATION = "ExponentiatedGradient(EqualizedOdds)"

show("AFTER mitigation — ExponentiatedGradient + EqualizedOdds", after)

print()
print("Primary Equalized-Odds diagnostics:")
print(f"  TPR difference (Equal opportunity diff): {after['Equal opportunity diff']:+.3f}")
print(f"  FPR difference                         : {after['False positive rate diff']:+.3f}")
print(f"  Equalized-odds gap                     : {after['Equalized odds gap']:.3f}")


In [ ]:
# --- Optional alternative: Reject Option Classification ---
# Not executed in this submission.
print("Reject Option Classification is not selected for this individual audit.")


## 8. Compare before vs after, and price the trade-off

This is the heart of the deliverable. A report that claims a fairness gain at *zero* cost usually did not measure carefully.

In [ ]:
compare = pd.DataFrame({"Before": before, "After": after})
compare["Change"] = compare["After"] - compare["Before"]
display(compare.round(3))

# Save actual values produced by THIS execution.
compare.to_csv("equalized_odds_before_after.csv", index=True)

print("Primary fairness objective: Equalized Odds")
print(
    f"  Equalized-odds gap: "
    f"{before['Equalized odds gap']:.3f} -> {after['Equalized odds gap']:.3f}"
)
print(
    f"  TPR gap: "
    f"{before['Equal opportunity diff']:+.3f} -> {after['Equal opportunity diff']:+.3f}"
)
print(
    f"  FPR gap: "
    f"{before['False positive rate diff']:+.3f} -> {after['False positive rate diff']:+.3f}"
)
print(
    f"  Accuracy: "
    f"{before['Accuracy']:.3f} -> {after['Accuracy']:.3f}"
)
print(
    f"  Disparate impact (secondary diagnostic): "
    f"{before['Disparate impact']:.3f} -> {after['Disparate impact']:.3f}"
)
print(
    "  Four-fifths:",
    fourfifths_flag(before["Disparate impact"]),
    "->",
    fourfifths_flag(after["Disparate impact"])
)


In [ ]:
# Before/after chart — metrics centered at zero
labels = [
    "TPR diff",
    "FPR diff",
    "Statistical parity diff",
    "Average odds diff",
]
keys = [
    "Equal opportunity diff",
    "False positive rate diff",
    "Statistical parity diff",
    "Average odds diff",
]

b = [before[k] for k in keys]
a = [after[k] for k in keys]

x = np.arange(len(labels))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w/2, b, w, label="Before")
ax.bar(x + w/2, a, w, label="After")
ax.axhline(0, color="black", lw=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=12, ha="right")
ax.set_ylabel("Difference (0 = parity)")
ax.set_title("Error-rate and parity gaps: before vs Equalized-Odds mitigation")
ax.legend()
plt.tight_layout()

CHART_FILE = "equalized_odds_before_after.png"
plt.savefig(CHART_FILE, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", CHART_FILE)


## 9. Is the change real, or is it noise?

Every number so far is a point estimate from **one** train/test split. Before you write "fairness improved" in a doctoral submission, check whether the improvement survives resampling. This cell repeats the whole pipeline across several splits and reports mean and standard deviation.

If a gap's mean is smaller than its standard deviation, you do not have a finding. You have noise. (Everyday picture: you cannot call a coin unfair after one toss.)

In [ ]:
N_SPLITS = 10
rows = []

for s in range(N_SPLITS):
    tr, te = dataset.split([1 - TEST_FRACTION], shuffle=True, seed=s)

    sc = StandardScaler().fit(tr.features)
    Xtr = sc.transform(tr.features)
    Xte = sc.transform(te.features)

    ytr_original = tr.labels.ravel()

    # BEFORE: ordinary baseline in original AIF360 coding.
    b_clf = LogisticRegression(max_iter=1000, random_state=s)
    b_clf.fit(Xtr, ytr_original)

    b_pred = make_pred_dataset(te, b_clf, Xte, FAV)
    b_m = audit(
        te, b_pred,
        unprivileged_groups,
        privileged_groups
    )

    # AFTER: Equalized-Odds in-processing.
    sens_tr = tr.protected_attributes[
        :, tr.protected_attribute_names.index(PROTECTED)
    ].ravel()

    # Fairlearn target: 1 = favorable.
    ytr_fav = (ytr_original == FAV).astype(int)

    eg_s = ExponentiatedGradient(
        estimator=LogisticRegression(max_iter=1000, random_state=s),
        constraints=EqualizedOdds(difference_bound=EO_BOUND),
        eps=EG_EPS,
    )
    eg_s.fit(
        Xtr,
        ytr_fav,
        sensitive_features=sens_tr
    )

    pred_fav_s = eg_s.predict(Xte, random_state=s)

    a_pred = make_fairlearn_pred_dataset(
        te,
        pred_fav_s,
        favorable_label=FAV,
        unfavorable_label=UNFAV
    )
    a_m = audit(
        te, a_pred,
        unprivileged_groups,
        privileged_groups
    )

    rows.append({
        "seed": s,
        **{f"before_{k}": v for k, v in b_m.items()},
        **{f"after_{k}":  v for k, v in a_m.items()}
    })

res = pd.DataFrame(rows)

summary = pd.DataFrame({
    "Before mean": [res[f"before_{k}"].mean() for k in before],
    "Before sd":   [res[f"before_{k}"].std()  for k in before],
    "After mean":  [res[f"after_{k}"].mean()  for k in before],
    "After sd":    [res[f"after_{k}"].std()   for k in before],
}, index=list(before))

summary["Mean change"] = summary["After mean"] - summary["Before mean"]

display(summary.round(3))

res.to_csv("equalized_odds_10_splits_raw.csv", index=False)
summary.to_csv("equalized_odds_10_splits_summary.csv", index=True)

print(f"Across {N_SPLITS} splits:")
for k in [
    "Equalized odds gap",
    "Equal opportunity diff",
    "False positive rate diff",
    "Accuracy",
    "Disparate impact",
]:
    mc = summary.loc[k, "Mean change"]
    spread = summary.loc[k, "After sd"]
    print(
        f"  {k:27s} mean change {mc:+.3f} | "
        f"after mean {summary.loc[k, 'After mean']:+.3f} | "
        f"after SD {spread:.3f}"
    )

print()
print("Interpretation rule:")
print("  Equalized Odds improves when BOTH TPR and FPR differences move closer to 0.")
print("  Disparate impact is still reported, but it is NOT the optimization objective.")
print("  The 10-split SDs are descriptive stability measures, not formal confidence intervals.")


## 10. Cross-check with Fairlearn, and the trap inside it

Using a second library to confirm a headline number is good scientific hygiene. But you cannot just hand your labels to Fairlearn and expect the same answer.

**Fairlearn treats label `1` as "selected".** In COMPAS the favorable label is `0`. (Everyday picture: two rulers, one marked in centimeters and one in inches. Both are correct, and you still cannot mix them.) So a naive call measures parity on the *unfavorable* outcome. You get away with it for the *difference* (it is symmetric up to sign, and Fairlearn returns max-minus-min so the sign is discarded), but the *ratio* comes out wrong and will not match AIF360's disparate impact. The cell below shows both, deliberately.

In [ ]:
# Independent inspection with Fairlearn MetricFrame.
# This is not a second mitigation; it is a second way to inspect group metrics.

from fairlearn.metrics import MetricFrame, true_positive_rate, false_positive_rate
from sklearn.metrics import accuracy_score

sens_test = test.protected_attributes[
    :, test.protected_attribute_names.index(PROTECTED)
].ravel()

y_true_f = (test.labels.ravel() == FAV).astype(int)
y_pred_f = (test_pred_mit.labels.ravel() == FAV).astype(int)

metrics_check = {
    "accuracy": accuracy_score,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
}

mf = MetricFrame(
    metrics=metrics_check,
    y_true=y_true_f,
    y_pred=y_pred_f,
    sensitive_features=sens_test,
)

print("Fairlearn group metrics after mitigation")
display(mf.by_group.round(3))

print()
print("AIF360 differences after mitigation")
print(f"TPR difference: {after['Equal opportunity diff']:+.3f}")
print(f"FPR difference: {after['False positive rate diff']:+.3f}")
print(f"Equalized-odds gap: {after['Equalized odds gap']:.3f}")


## 11. Generate the Bias Audit Report from the results you actually obtained

The next cell creates the report **only from variables computed during this run**. No metric value is written in advance. It saves a Markdown report based on your Equalized-Odds experiment.


In [ ]:
# --- Generate the individual report from THIS execution only ---
from pathlib import Path

raw_di = float(raw.disparate_impact())
raw_spd = float(raw.mean_difference())

priv_label = GROUP_LABELS[PROTECTED][1]
unpriv_label = GROUP_LABELS[PROTECTED][0]

before_di_flag = fourfifths_flag(before["Disparate impact"])
after_di_flag  = fourfifths_flag(after["Disparate impact"])

# Did the primary Equalized-Odds components improve?
tpr_before = abs(before["Equal opportunity diff"])
tpr_after = abs(after["Equal opportunity diff"])
fpr_before = abs(before["False positive rate diff"])
fpr_after = abs(after["False positive rate diff"])
eo_gap_before = before["Equalized odds gap"]
eo_gap_after = after["Equalized odds gap"]

if eo_gap_after < eo_gap_before:
    primary_verdict = "The Equalized-Odds gap decreased on the main split."
elif eo_gap_after > eo_gap_before:
    primary_verdict = "The Equalized-Odds gap increased on the main split; the mitigation did not improve the primary objective on this split."
else:
    primary_verdict = "The Equalized-Odds gap was unchanged on the main split."

acc_change_mean = summary.loc["Accuracy", "Mean change"]
eo_gap_change_mean = summary.loc["Equalized odds gap", "Mean change"]
tpr_change_mean = summary.loc["Equal opportunity diff", "Mean change"]
fpr_change_mean = summary.loc["False positive rate diff", "Mean change"]

report = f"""# Bias Audit Report — Individual Equalized-Odds Variant
**Student:** {STUDENT_NAME}

## 1. Setup
- **Dataset:** ProPublica COMPAS two-year recidivism dataset loaded through AIF360.
- **Protected attribute:** `{PROTECTED}`
- **Privileged group:** {priv_label}
- **Unprivileged group:** {unpriv_label}
- **Favorable outcome:** did NOT re-offend within two years.
- **Favorable label code:** `{FAV}`
- **Baseline:** Logistic Regression.
- **Individual mitigation:** ExponentiatedGradient + EqualizedOdds.
- **Equalized-Odds optimization bound:** `{EO_BOUND}`.
- **Scientific difference from the exemplar:** the exemplar prioritizes statistical parity with Reweighing; this submission prioritizes error-rate balance.

## 2. Bias already in the labels
- **Disparate impact in training labels:** `{raw_di:.3f}`
- **Statistical parity difference in training labels:** `{raw_spd:+.3f}`

Historical group disparity exists before model training. This is important context, but my mitigation objective is not statistical parity; it is Equalized Odds.

## 3. BEFORE metrics

| Metric | Value |
|---|---:|
| Accuracy | {before["Accuracy"]:.3f} |
| Disparate impact | {before["Disparate impact"]:.3f} |
| Statistical parity difference | {before["Statistical parity diff"]:+.3f} |
| TPR difference / equal opportunity difference | {before["Equal opportunity diff"]:+.3f} |
| FPR difference | {before["False positive rate diff"]:+.3f} |
| Equalized-Odds gap | {before["Equalized odds gap"]:.3f} |
| Average odds difference | {before["Average odds diff"]:+.3f} |

The primary baseline concern for this individual audit is the combination of TPR and FPR differences, summarized by the Equalized-Odds gap.

## 4. Mitigation applied and why
I use **ExponentiatedGradient with an EqualizedOdds constraint**, an in-processing mitigation. I chose this objective because a recidivism-risk model can harm groups through unequal false alarms as well as unequal misses. Equalized Odds therefore aligns more directly with the distribution of classification errors than statistical parity alone.

Fairlearn expects binary classification with `1` as the positive class, whereas AIF360 COMPAS defines the favorable class as `0`. For the mitigation only, I recoded `favorable -> 1` and `unfavorable -> 0`; after prediction I mapped the outputs back to COMPAS coding before calculating AIF360 metrics.

## 5. AFTER metrics and trade-off

| Metric | Before | After | Change |
|---|---:|---:|---:|
| Accuracy | {before["Accuracy"]:.3f} | {after["Accuracy"]:.3f} | {after["Accuracy"]-before["Accuracy"]:+.3f} |
| Disparate impact | {before["Disparate impact"]:.3f} | {after["Disparate impact"]:.3f} | {after["Disparate impact"]-before["Disparate impact"]:+.3f} |
| Statistical parity difference | {before["Statistical parity diff"]:+.3f} | {after["Statistical parity diff"]:+.3f} | {after["Statistical parity diff"]-before["Statistical parity diff"]:+.3f} |
| TPR difference | {before["Equal opportunity diff"]:+.3f} | {after["Equal opportunity diff"]:+.3f} | {after["Equal opportunity diff"]-before["Equal opportunity diff"]:+.3f} |
| FPR difference | {before["False positive rate diff"]:+.3f} | {after["False positive rate diff"]:+.3f} | {after["False positive rate diff"]-before["False positive rate diff"]:+.3f} |
| Equalized-Odds gap | {before["Equalized odds gap"]:.3f} | {after["Equalized odds gap"]:.3f} | {after["Equalized odds gap"]-before["Equalized odds gap"]:+.3f} |
| Average odds difference | {before["Average odds diff"]:+.3f} | {after["Average odds diff"]:+.3f} | {after["Average odds diff"]-before["Average odds diff"]:+.3f} |

**Primary-objective reading:** {primary_verdict}

**Secondary disparate-impact check:** {before_di_flag} -> {after_di_flag}. This is reported for transparency, but disparate impact was not the optimization target.

## 6. Uncertainty across 10 splits

| Metric | Before mean ± SD | After mean ± SD | Mean change |
|---|---:|---:|---:|
| Accuracy | {summary.loc["Accuracy","Before mean"]:.3f} ± {summary.loc["Accuracy","Before sd"]:.3f} | {summary.loc["Accuracy","After mean"]:.3f} ± {summary.loc["Accuracy","After sd"]:.3f} | {summary.loc["Accuracy","Mean change"]:+.3f} |
| Equalized-Odds gap | {summary.loc["Equalized odds gap","Before mean"]:.3f} ± {summary.loc["Equalized odds gap","Before sd"]:.3f} | {summary.loc["Equalized odds gap","After mean"]:.3f} ± {summary.loc["Equalized odds gap","After sd"]:.3f} | {summary.loc["Equalized odds gap","Mean change"]:+.3f} |
| TPR difference | {summary.loc["Equal opportunity diff","Before mean"]:+.3f} ± {summary.loc["Equal opportunity diff","Before sd"]:.3f} | {summary.loc["Equal opportunity diff","After mean"]:+.3f} ± {summary.loc["Equal opportunity diff","After sd"]:.3f} | {summary.loc["Equal opportunity diff","Mean change"]:+.3f} |
| FPR difference | {summary.loc["False positive rate diff","Before mean"]:+.3f} ± {summary.loc["False positive rate diff","Before sd"]:.3f} | {summary.loc["False positive rate diff","After mean"]:+.3f} ± {summary.loc["False positive rate diff","After sd"]:.3f} | {summary.loc["False positive rate diff","Mean change"]:+.3f} |
| Disparate impact | {summary.loc["Disparate impact","Before mean"]:.3f} ± {summary.loc["Disparate impact","Before sd"]:.3f} | {summary.loc["Disparate impact","After mean"]:.3f} ± {summary.loc["Disparate impact","After sd"]:.3f} | {summary.loc["Disparate impact","Mean change"]:+.3f} |

The repeated-split analysis determines whether the apparent Equalized-Odds improvement is stable rather than a lucky split. The mean accuracy change is `{acc_change_mean:+.3f}`. The mean Equalized-Odds-gap change is `{eo_gap_change_mean:+.3f}`. These standard deviations are descriptive stability measures, not formal confidence intervals.

## 7. Recommendation
The deployment recommendation must be based primarily on the achieved TPR/FPR balance, not on disparate impact alone. If Equalized Odds improves consistently across splits at an acceptable accuracy cost, the mitigation is a defensible candidate for further evaluation. It is still not sufficient evidence for high-stakes deployment without jurisdiction-specific validation, calibration analysis, and intersectional subgroup audits.

## 8. Honest limitation
This experiment deliberately changes the fairness objective, but it still audits only race marginally and uses only one mitigation family. ExponentiatedGradient also produces a randomized ensemble, which improves constraint satisfaction at the cost of a less straightforward individual-level explanation than a single deterministic Logistic Regression. I therefore cannot claim that this method is universally preferable to Reweighing; only that it tests a different and defensible fairness objective.

## Transfer to my doctoral research
The methodological lesson is directly relevant to my future XAI study of school-interruption risk in Peru: I should not ask whether a model is simply “fair.” I must first identify the most serious harm in the educational context, choose a fairness metric that corresponds to that harm, report subgroup error rates, and quantify the predictive trade-off across repeated runs.

---
*All numerical values above are generated from this notebook execution. No result is copied from the professor's exemplar.*
"""

print(report)

REPORT_FILE = "Bias_Audit_Report_Cendik_EqualizedOdds.md"
Path(REPORT_FILE).write_text(report, encoding="utf-8")
print("\\nSaved:", REPORT_FILE)


## Appendix A. Optional alternative dataset: Adult Census income

This appendix is **not part of the selected COMPAS submission** and is disabled by default. It uses a separate variable (`adult_dataset`) and cannot overwrite the COMPAS state used by the report.


In [ ]:
# Optional Adult loader — disabled for the COMPAS assignment.
if RUN_ADULT_ALTERNATIVE:
    from aif360.datasets import StandardDataset
    from fairlearn.datasets import fetch_adult

    def load_adult():
        d = fetch_adult(as_frame=True)
        df = d.data.copy()
        df["income"] = (d.target.astype(str).str.contains(">50K")).astype(int)
        df = df.dropna()
        return StandardDataset(
            df=df,
            label_name="income",
            favorable_classes=[1],
            protected_attribute_names=["sex", "race"],
            privileged_classes=[["Male"], ["White"]],
            categorical_features=[
                "workclass", "education", "marital-status",
                "occupation", "relationship", "native-country"
            ],
            features_to_drop=["fnlwgt"]
        )

    adult_dataset = load_adult()
    print("Adult alternative loaded separately as `adult_dataset`.")
    print("It does NOT overwrite the COMPAS variables used in this assignment.")
else:
    print("Adult alternative skipped (RUN_ADULT_ALTERNATIVE = False).")


## 12. Package the Equalized-Odds evidence files

After a successful run, this cell creates a ZIP containing the report, actual result tables, and chart from **your execution**.


In [ ]:
import os, zipfile

files_to_package = [
    "Bias_Audit_Report_Cendik_EqualizedOdds.md",
    "equalized_odds_before_after.csv",
    "equalized_odds_10_splits_raw.csv",
    "equalized_odds_10_splits_summary.csv",
    "equalized_odds_before_after.png",
]

existing = [f for f in files_to_package if os.path.exists(f)]
ZIP_FILE = "Session12_Bias_Audit_Cendik_EqualizedOdds_Results.zip"

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as z:
    for f in existing:
        z.write(f)

print("Created:", ZIP_FILE)
print("Included:")
for f in existing:
    print(" -", f)

# Optional Google Colab download:
# from google.colab import files
# files.download(ZIP_FILE)
